## 🤖 ML Models - Crime Pattern Intelligence
This notebook builds 3 machine learning models using Spark MLlib tracked with MLflow:
- **Model 1:** Crime Type Classifier (what type of crime will occur?)
- **Model 2:** Arrest Likelihood Scorer (will an arrest be made?)
- **Model 3:** Hotspot Predictor (which district will be hit next?)

In [0]:
import mlflow
import mlflow.spark
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# Load Silver table
df = spark.read.format("delta").table("workspace.default.silver_crimes")

# Set MLflow experiment
mlflow.set_experiment("/crime_intelligence_models")

print(f"✅ Data loaded: {df.count()} records")
print(f"✅ MLflow experiment set!")

✅ Data loaded: 1269490 records
✅ MLflow experiment set!


### 🎯 Model 1 - Arrest Likelihood Scorer
Predict whether an arrest will be made based on crime type, location, time and other features. This is a binary classification problem (Arrest: true/false).

In [0]:
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from mlflow.models.signature import infer_signature
import os

os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/crime_data"

indexer_primary = StringIndexer(inputCol="Primary_Type", outputCol="Primary_Type_Idx", handleInvalid="keep")
indexer_location = StringIndexer(inputCol="Location_Description", outputCol="Location_Idx", handleInvalid="keep")
indexer_season = StringIndexer(inputCol="Season", outputCol="Season_Idx", handleInvalid="keep")

df_model1 = df.withColumn("label", df["Arrest"].cast("integer"))

assembler = VectorAssembler(inputCols=[
    "Primary_Type_Idx", "Location_Idx", "Season_Idx",
    "Hour", "Day_Of_Week", "Month", "District",
    "Is_Weekend", "Is_Night", "Domestic"
], outputCol="features", handleInvalid="keep")

train, test = df_model1.randomSplit([0.8, 0.2], seed=42)

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=10)
pipeline = Pipeline(stages=[indexer_primary, indexer_location, indexer_season, assembler, lr])

with mlflow.start_run(run_name="arrest_likelihood_scorer") as run:
    model = pipeline.fit(train)
    predictions = model.transform(test)
    
    evaluator = BinaryClassificationEvaluator(labelCol="label")
    auc = evaluator.evaluate(predictions)
    
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("features", "Primary_Type, Location, Season, Hour, Day_Of_Week, Month, District, Is_Weekend, Is_Night, Domestic")
    mlflow.log_metric("AUC", auc)

    sample_input = train.limit(5)
    sample_output = model.transform(sample_input).limit(5)

    signature = infer_signature(
        sample_input.toPandas(),
        sample_output.toPandas()
    )

    mlflow.spark.log_model(
        spark_model=model,
        artifact_path="arrest_likelihood_model",
        signature=signature
    )
    
    print(f"✅ Model 1 - Arrest Likelihood Scorer")
    print(f"   AUC Score: {round(auc, 4)}")
    print(f"   Run ID: {run.info.run_id}")
    print(f"   Logged to MLflow with signature!")

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/20 06:19:16 WARNING mlflow.models.signature: Failed to infer schema for outputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. To see the full traceback, set logging level to DEBUG.
2026/05/20 06:19:39 WARNING mlf

✅ Model 1 - Arrest Likelihood Scorer
   AUC Score: 0.6896
   Run ID: 317da5b60be841e2b220557c365dc0df
   Logged to MLflow with signature!


### 🗺️ Model 2 - Crime Type Classifier
Predict what type of crime is likely to occur based on location, time and district. This is a multi-class classification problem (31 crime types).

In [0]:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from mlflow.models.signature import infer_signature

# Index the target label (Primary_Type)
indexer_label = StringIndexer(inputCol="Primary_Type", outputCol="label", handleInvalid="keep")

# Prepare features
indexer_location2 = StringIndexer(inputCol="Location_Description", outputCol="Location_Idx2", handleInvalid="keep")
indexer_season2 = StringIndexer(inputCol="Season", outputCol="Season_Idx2", handleInvalid="keep")

assembler2 = VectorAssembler(inputCols=[
    "Location_Idx2", "Season_Idx2",
    "Hour", "Day_Of_Week", "Month", 
    "District", "Is_Weekend", "Is_Night", "Domestic"
], outputCol="features2", handleInvalid="keep")

# Train/test split
train2, test2 = df.randomSplit([0.8, 0.2], seed=42)

# Fix: increase maxBins to handle 150 location categories
rf = RandomForestClassifier(
    featuresCol="features2", 
    labelCol="label", 
    numTrees=50,
    maxBins=256
)

pipeline2 = Pipeline(stages=[
    indexer_label,
    indexer_location2,
    indexer_season2,
    assembler2,
    rf
])

# Train with MLflow tracking
with mlflow.start_run(run_name="crime_type_classifier") as run:

    model2 = pipeline2.fit(train2)
    predictions2 = model2.transform(test2)
    
    evaluator2 = MulticlassClassificationEvaluator(
        labelCol="label",
        metricName="accuracy"
    )

    accuracy = evaluator2.evaluate(predictions2)
    
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("num_trees", 50)
    mlflow.log_param("maxBins", 256)
    mlflow.log_metric("accuracy", accuracy)

    # Signature
    sample_input = train2.limit(5)
    sample_output = model2.transform(sample_input).limit(5)

    signature = infer_signature(
        sample_input.toPandas(),
        sample_output.toPandas()
    )

    mlflow.spark.log_model(
        spark_model=model2,
        artifact_path="crime_type_classifier",
        signature=signature
    )
    
    print(f"✅ Model 2 - Crime Type Classifier")
    print(f"   Accuracy: {round(accuracy * 100, 2)}%")
    print(f"   Run ID: {run.info.run_id}")
    print(f"   Logged to MLflow with signature!")

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/20 06:22:28 WARNING mlflow.models.signature: Failed to infer schema for outputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. To see the full traceback, set logging level to DEBUG.
2026/05/20 06:22:46 WARNING mlf

✅ Model 2 - Crime Type Classifier
   Accuracy: 31.4%
   Run ID: a912d51c3c504f2d8b722168151dd3bd
   Logged to MLflow with signature!


### 🗺️ Model 3 - Crime Hotspot Predictor
Predict whether a district will be a high crime hotspot based on time features. Binary classification — top 5 busiest districts vs rest.

In [0]:
# Clear old Python references only
for var in [
    "model", "model2", "model3",
    "predictions", "predictions2", "predictions3",
    "pipeline", "pipeline2", "pipeline3",
    "train", "test", "train2", "test2", "train3", "test3"
]:
    if var in globals():
        del globals()[var]

import gc
gc.collect()

print("✅ Cleared Python model references")

✅ Cleared Python model references


In [0]:
from pyspark.sql.functions import when, col
from mlflow.models.signature import infer_signature

# Create hotspot label — top 5 districts by crime volume are "hotspots" (label=1)
hotspot_districts = [8, 12, 6, 4, 11]

df_model3 = df.withColumn(
    "label",
    when(col("District").isin(hotspot_districts), 1).otherwise(0)
)

# Features
indexer_season3 = StringIndexer(
    inputCol="Season",
    outputCol="Season_Idx3",
    handleInvalid="keep"
)

assembler3 = VectorAssembler(inputCols=[
    "Season_Idx3",
    "Hour",
    "Day_Of_Week",
    "Month",
    "Is_Weekend",
    "Is_Night"
], outputCol="features3", handleInvalid="keep")

# Train/test split
train3, test3 = df_model3.randomSplit([0.8, 0.2], seed=42)

# Logistic Regression for hotspot prediction
lr3 = LogisticRegression(
    featuresCol="features3",
    labelCol="label",
    maxIter=20
)

pipeline3 = Pipeline(stages=[
    indexer_season3,
    assembler3,
    lr3
])

# Train with MLflow tracking
with mlflow.start_run(run_name="crime_hotspot_predictor") as run:

    model3 = pipeline3.fit(train3)
    predictions3 = model3.transform(test3)
    
    evaluator3 = BinaryClassificationEvaluator(labelCol="label")
    auc3 = evaluator3.evaluate(predictions3)
    
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("hotspot_districts", hotspot_districts)
    mlflow.log_param("maxIter", 20)
    mlflow.log_metric("AUC", auc3)

    # Signature
    sample_input = train3.limit(5)
    sample_output = model3.transform(sample_input).limit(5)

    signature = infer_signature(
        sample_input.toPandas(),
        sample_output.toPandas()
    )

    mlflow.spark.log_model(
        spark_model=model3,
        artifact_path="crime_hotspot_predictor",
        signature=signature
    )
    
    print(f"✅ Model 3 - Crime Hotspot Predictor")
    print(f"   AUC Score: {round(auc3, 4)}")
    print(f"   Hotspot Districts: {hotspot_districts}")
    print(f"   Run ID: {run.info.run_id}")
    print(f"   Logged to MLflow with signature!")

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/05/20 06:27:13 WARNING mlflow.models.signature: Failed to infer schema for outputs. Setting schema to `Schema([ColSpec(type=AnyType())]` as default. To see the full traceback, set logging level to DEBUG.
2026/05/20 06:27:23 WARNING mlf

✅ Model 3 - Crime Hotspot Predictor
   AUC Score: 0.5119
   Hotspot Districts: [8, 12, 6, 4, 11]
   Run ID: 73745bf9cd4d47f08ff67f15e65849f3
   Logged to MLflow with signature!
